# Lesson 20 Lab — FP8, FP4, NVFP4, and Hardware Boundaries

**Puzzle:** Does Blackwell hardware support mean every framework build exposes the same FP8 or NVFP4 path?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A dtype name can exist at four levels: a mathematical format, a hardware instruction, a library recipe, and a framework operator. Blackwell support does not guarantee that the installed PyTorch, Transformer Engine, TensorRT, or ModelOpt build exposes the same FP8 or NVFP4 path. Each layer must be probed independently.


## 0. Predict before running

1. Distinguish E4M3 FP8 from E5M2 and ordinary INT4 from block-scaled NVFP4.
2. Predict whether the installed PyTorch build can execute a scaled FP8 matrix multiply.
3. State what additional evidence is needed before claiming NVFP4 performance.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Keep four layers distinct: numerical format, hardware instruction, library recipe, and framework/operator API. `torch.float8_*` existing does not alone prove an FP8 GEMM path.

- A format definition, hardware instruction, library API, and framework kernel are four separate layers.
- FP8 variants trade exponent range against fraction precision.
- NVFP4 adds block scaling; it is not ordinary uniform INT4.


## 2. Derive the mechanism

E4M3 favors precision with less range; E5M2 favors range. Scaled FP8 matmul applies explicit scale factors. Blackwell-specific MXFP8/NVFP4 add block-scale structure and require matching recipes and kernels.

FP8 E4M3 allocates four exponent and three fraction bits after sign, trading range for precision; E5M2 spends another bit on range. NVFP4 uses FP4 E2M1 values with block scaling, so its real representation includes both four-bit data and scale hierarchy. TensorRT's current scheme uses block size 16 for NVFP4, while framework APIs and supported axes remain version-specific.

Scaled matrix multiplication also requires choosing input and output scales. A successful `torch._scaled_mm` call proves one framework-level path for one shape and format; it does not prove Transformer Engine recipes or TensorRT NVFP4 kernels.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "20-fp8-fp4-nvfp4"
device = require_cuda()
torch.manual_seed(2026 + 20)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | higher-precision reference matrix multiplication for error comparison |
| Candidate | PyTorch scaled FP8 E4M3 GEMM on RTX 5090 |
| Held constant | 1024-class matrix shape, scaling procedure, warm-up, fifteen timing samples |
| Measurements | API success, RMSE/cosine, median/p90, library availability, NVFP4 status |
| Evidence | `pytorch-gpu` |

**Experiment:** Attempt native PyTorch FP8 GEMM on the RTX GPU, record error and timing when supported, and separately probe Transformer Engine and NVFP4 APIs.


## 5. Read the experiment code

The lab calls PyTorch scaled FP8 matmul when available and leaves Transformer Engine/NVFP4 unmeasured rather than equating hardware generation with framework support.

The notebook checks for float8 dtype support and calls `torch._scaled_mm` with explicit scales. It compares the output with a higher-precision reference and times repeated CUDA execution. Separate probes record Transformer Engine availability and leave NVFP4 `not_measured` when its recipe/operator is unavailable.

This design prevents the real FP8 result from being generalized to a different format. The JSON names the exact API so a future software change can be detected.

Only after these variables match the protocol should the cell be executed.


In [2]:
import importlib.util
n=1024; a=torch.randn(n,n,device=device,dtype=torch.bfloat16); b=torch.randn(n,n,device=device,dtype=torch.bfloat16); ref=(a@b).float()
probe={"torch_float8_dtype":hasattr(torch,"float8_e4m3fn"),"transformer_engine_installed":importlib.util.find_spec("transformer_engine") is not None}
if probe["torch_float8_dtype"]:
    try:
        a8=a.to(torch.float8_e4m3fn); b8=b.to(torch.float8_e4m3fn)
        one=torch.tensor(1.0,device=device)
        def fp8_scaled_mm():
            return torch._scaled_mm(a8,b8,scale_a=one,scale_b=one,out_dtype=torch.bfloat16)
        out=fp8_scaled_mm().float()
        probe.update({"fp8_gemm":"success","api":"torch._scaled_mm","fp8_error":error_metrics(ref,out),
                      "fp8_timing":cuda_benchmark(fp8_scaled_mm,warmup=5,repeats=15)})
    except Exception as exc: probe.update({"fp8_gemm":"failed","error_type":type(exc).__name__,"error_message":str(exc)[:240]})
probe["nvfp4_backend"]="not_measured"
result=base_result(20,"pytorch-gpu" if probe.get("fp8_gemm")=="success" else "compatibility-probe"); result.update({"probe":probe,
    "conclusion":"Framework-level FP8 was tested independently; NVFP4 requires a supported library recipe and operator evidence."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| PyTorch API | torch._scaled_mm |
| FP8 GEMM | success |
| Median | 0.017568 ms |
| FP8 RMSE | 1.208455 |
| Transformer Engine installed | no |
| NVFP4 backend | not_measured |


## 7. Interpret rather than merely print

The scaled FP8 GEMM succeeded through `torch._scaled_mm`, with median 0.017568 ms and p90 0.018560 ms over fifteen samples. Output cosine was 0.999285 and RMSE 1.208455 for the tested scale and shape. Transformer Engine was not installed, and NVFP4 remained `not_measured`.

The measured path is therefore real PyTorch GPU evidence for FP8, not proof of a Transformer Engine or NVFP4 backend. The absolute error also shows why format support must be paired with scaling and quality policy.

**Inspection rule:** A successful float8 PyTorch GEMM proves that path only. NVFP4 remains unmeasured without its library recipe and operator evidence.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Framework-level FP8 was tested independently; NVFP4 requires a supported library recipe and operator evidence.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:01+00:00",
  "lesson": 20,
  "probe": {
    "api": "torch._scaled_mm",
    "fp8_error": {
      "cosine": 0.99928468,
      "mae": 0.96194851,
      "max_abs": 6.1875,
      "rmse": 1.20845497
    },
    "fp8_gemm": "success",
    "fp8_timing": {
      "median_ms": 0.017568,
      "p90_ms": 0.01856,
      "repeats": 15,
      "samples_ms": [
        0.028544,
        0.020288,
        0.01856,
        0.017568,
        0.017568,
        0.01824,
        0.017312,
        0.01728,
        0.016928,
        0.0176,
        0.017504,
        0.016864,
        0.017824,
        0.017216,
  

## 9. Make the bounded decision

> Publish a format-by-hardware-by-library matrix, not a single `supported` checkbox.

**Acceptance/rollback:** Record compute capability, dtype/API, scaling recipe, operator success, numerical error, timing, and library version separately for FP8, MXFP8, and NVFP4.

**Failure analysis:** Casting tensors to a float8 dtype without a successful matrix operator proves storage only. Comparing raw FP8 latency against a different shape or excluding scale computation can misstate speed. Treating NVFP4 as signed uniform INT4 loses its block-scale semantics entirely.


## 10. Extend the evidence

Install a matching Transformer Engine or TensorRT stack in isolation, run documented FP8 and NVFP4 recipes, and capture operator identity, scale granularity, end-to-end scale overhead, error, and latency. Build a matrix with rows for format and columns for hardware, library, API, operator, and tested status.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
